In [1]:
import pandas as pd
from datasets import Dataset
import torch

# Load dataset
df = pd.read_csv("news_data_1M_lines.txt", sep="\t", header=None, on_bad_lines="skip").head(60000)
df.columns = ["incorrect", "correct"]
df = df.dropna().drop_duplicates()
df["incorrect"] = df["incorrect"].str.strip()
df["correct"] = df["correct"].str.strip()

print("Samples:", len(df))
display(df.head())

dataset = Dataset.from_pandas(df)

Samples: 59951


,incorrect,correct
0,"प्रतोनध, नगपुर पन ठेे बद असतनह सध्य मोठ्य प्रम...","प्रतिनिधी, नागपूर पान ठेले बंद असतानाही सध्या ..."
1,"मत्र, ह खर्र तयर करण्यसठि बूरश चंढलेल्य सूपरिच...","मात्र, हा खर्रा तयार करण्यासाठी बुरशी चढलेल्या..."
2,बद असतन गटखह नगपरत मोठ्य प्रमणत यते असल्यचंेहि...,बंदी असताना गुटखाही नागपुरात मोठ्या प्रमाणात य...
3,अअन्न वा षऔध प्रशसन वीभगने केलाल्य तपसणत ७ु१ न...,अन्न व औषध प्रशासन विभागाने केलेल्या तपासणीत ७...
4,"म्य प्दरेश, छत्तिसगढ, आध्र प्रेदश य भगतुन मल ब...","मध्य प्रदेश, छत्तीसगढ, आंध्र प्रदेश या भागातून..."


In [2]:
import re

def clean_marathi_text(text):
    # Remove sequences like NU123, NU१२३, or NMU sequences (any letters followed by digits)
    text = re.sub(r"[A-Za-z]*[0-9०-९]+", "", text)

    # Keep: full Devanagari block, Latin letters (optional), digits, spaces, basic punctuation
    text = re.sub(r"[^\u0900-\u097F\s.,।'’\"]+", "", text)

    # Replace multiple spaces with a single space
    text = re.sub(r"\s+", " ", text)

    return text.strip()

# Apply to dataset
df["incorrect"] = df["incorrect"].apply(clean_marathi_text)
df["correct"] = df["correct"].apply(clean_marathi_text)

display(df.head())

,incorrect,correct
0,"प्रतोनध, नगपुर पन ठेे बद असतनह सध्य मोठ्य प्रम...","प्रतिनिधी, नागपूर पान ठेले बंद असतानाही सध्या ..."
1,"मत्र, ह खर्र तयर करण्यसठि बूरश चंढलेल्य सूपरिच...","मात्र, हा खर्रा तयार करण्यासाठी बुरशी चढलेल्या..."
2,बद असतन गटखह नगपरत मोठ्य प्रमणत यते असल्यचंेहि...,बंदी असताना गुटखाही नागपुरात मोठ्या प्रमाणात य...
3,अअन्न वा षऔध प्रशसन वीभगने केलाल्य तपसणत ु नमू...,अन्न व औषध प्रशासन विभागाने केलेल्या तपासणीत न...
4,"म्य प्दरेश, छत्तिसगढ, आध्र प्रेदश य भगतुन मल ब...","मध्य प्रदेश, छत्तीसगढ, आंध्र प्रदेश या भागातून..."


In [3]:
# Check for empty rows after cleaning
print("Empty incorrect rows:", df["incorrect"].isna().sum())
print("Empty correct rows:", df["correct"].isna().sum())

# Drop empty rows if any
df = df.dropna().reset_index(drop=True)

# Check a few samples
print(df.sample(5))

Empty incorrect rows: 0
Empty correct rows: 0
                                               incorrect  \
41167  अमेरीकेचंे रष्ट्रध्यक्ष डोनल्ड ट्रमू्प यँनि लै...   
26884  करोन वषणुचं्य वढत्य सँक्रमणमळे सर्वसमन्यँसमोर ...   
25206  ि ः ः करोन स, उत्सवँबबत अजीत पवर यचंे महत्त्वच...   
34707  हचं वेेळ सधुन मगळवरि सयँकळ य मलवर बलत्कर करण्य...   
25602  प्रतनीधि, कोल्हपुर जील्ह्यत करोनचं हकर वढत असत...   

                                                 correct  
41167  अमेरिकेचे राष्ट्राध्यक्ष डोनाल्ड ट्रम्प यांनी ...  
26884  करोना विषाणूच्या वाढत्या संक्रमणामुळे सर्वसामा...  
25206  करोना सण, उत्सवांबाबत अजित पवार यांचे महत्त्वा...  
34707  हीच वेळ साधून मंगळवारी सायंकाळी या मुलीवर बलात...  
25602  प्रतिनिधी, कोल्हापूरः जिल्ह्यात करोनाचा कहर वा...  


In [4]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))

Train samples: 53955
Validation samples: 5996


In [5]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

print(train_dataset[0])

{'incorrect': 'घरबहेर जयेूश तचं वट बघतत बसल होत.', 'correct': 'घराबाहेर जयेश तिची वाट बघत बसला होता.', '__index_level_0__': 11376}


In [6]:
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM

MODEL_CHECKPOINT = "ai4bharat/indicbart"  # IndicBART model
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)

# Tokenize function
def tokenize(batch):
    return tokenizer(batch["incorrect"], text_target=batch["correct"],
                     truncation=True, padding="max_length", max_length=64)

# Apply tokenization
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

# Remove unnecessary columns
train_dataset = train_dataset.remove_columns(["incorrect", "correct", "__index_level_0__"])
val_dataset = val_dataset.remove_columns(["incorrect", "correct", "__index_level_0__"])

Map:   0%|          | 0/53955 [00:00<?, ? examples/s]

Map:   0%|          | 0/5996 [00:00<?, ? examples/s]

In [7]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [8]:
print(train_dataset[0]['labels'])
print(sum([1 for i in train_dataset[0]['labels'] if i != tokenizer.pad_token_id]))

[2, 401, 26009, 1910, 820, 28371, 4563, 32684, 21008, 538, 6, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
12


In [9]:
train_dataset_small = train_dataset.select(range(500))
val_dataset_small = val_dataset.select(range(100))

In [10]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, AutoModelForSeq2SeqLM
from transformers import DataCollatorForSeq2Seq
from torch.utils.data import DataLoader
import torch

# ✅ Step 1: Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=False,  
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)


trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)



trainer.train()
model.save_pretrained("lang_analyzer_model")
tokenizer.save_pretrained("lang_analyzer_model")

C:\Users\samar\AppData\Local\Temp\ipykernel_11764\2034901879.py:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 3, 'bos_token_id': 2}.


Epoch,Training Loss,Validation Loss
1,0.714100,0.552311
2,0.577600,0.465179
3,0.524100,0.444595


('lang_analyzer_model\\tokenizer_config.json',
 'lang_analyzer_model\\special_tokens_map.json',
 'lang_analyzer_model\\spiece.model',
 'lang_analyzer_model\\added_tokens.json',
 'lang_analyzer_model\\tokenizer.json')

In [13]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Load trained model + tokenizer
MODEL_DIR = "lang_analyzer_model"   # your saved model folder
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR)

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()  # ⚡ set to evaluation mode for speed

# --- Function for correcting a single sentence ---
def correct_sentence(incorrect_sentence, max_length=64):
    # Tokenize
    inputs = tokenizer(
        incorrect_sentence,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    # 🚫 Remove token_type_ids (MBart/IndicBART doesn’t use them)
    if "token_type_ids" in inputs:
        del inputs["token_type_ids"]

    # Send to GPU/CPU
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Fast greedy decoding (⚡ <1 sec)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_beams=1,       # greedy search (faster than beam search)
        do_sample=False
    )

    # Decode prediction
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# --- Example ---
incorrect_sentence = "य प्रकरणि न्ययलयत दोषरोपपत्र दखल करण्यत आआले आहेु."
corrected = correct_sentence(incorrect_sentence)

print("Incorrect:", incorrect_sentence)
print("Corrected:", corrected)


Incorrect: य प्रकरणि न्ययलयत दोषरोपपत्र दखल करण्यत आआले आहेु.
Corrected: या प्रकरणी न्यायालयात दोषारोपपत्र दाखल करण्यात आले आहे.


In [23]:
from evaluate import load
import torch
from tqdm import tqdm  # progress bar

sacrebleu = load("sacrebleu")

model.eval()
predictions = []
references = []

max_length = 64

# Loop with progress bar
for example in tqdm(val_dataset.select(range(1000)), desc="Generating predictions"):
    # Convert input_ids and attention_mask to tensors
    input_ids = torch.tensor(example["input_ids"], dtype=torch.long).unsqueeze(0).to(device)
    attention_mask = torch.tensor(example["attention_mask"], dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=max_length,
            num_beams=4
        )

    # Decode prediction
    corrected_sentence = tokenizer.decode(outputs[0], skip_special_tokens=True)
    predictions.append(corrected_sentence)

    # Prepare reference sentence
    labels = [id for id in example["labels"] if id != -100]
    reference_sentence = tokenizer.decode(labels, skip_special_tokens=True)
    references.append([reference_sentence])  # sacrebleu expects list of lists

# Compute BLEU score
results = sacrebleu.compute(predictions=predictions, references=references)
print("SacreBLEU score:", results["score"])

Generating predictions: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [17:30<00:00,  1.05s/it]


SacreBLEU score: 30.930598442650812
